# Multi-provider calls, sampling controls, and temperature
**The messages format · temperature, top_p, max_tokens, stop · switching provider by editing .env**

Run top to bottom. Before starting: `git pull`, `uv sync` (no new packages today).

Two providers need a key: Groq (already in your `.env`) and Gemini (new today — see section 1). Ollama is optional; cells that need it skip cleanly if it is not running.

## 1. Setup — three clients, one function

A provider is a place; a model is a brain. `ask()` in `providers_utils.py` is the one function every provider boils down to: build a messages list, call `chat.completions.create`, read the text and the usage. Only the client and the model name change between providers.

**Getting a free Gemini key (2 minutes):** go to aistudio.google.com, sign in, click "Get API key", create one, and paste it into `.env` as `GEMINI_API_KEY`. No card needed.

In [6]:
import sys; sys.path[:0] = [".", "module01", "../module01"]
import os
from providers_utils import ask, classify_ticket, gemini_client, groq_client, list_gemini_models, ollama_client
from tickets_data import ALL_TICKETS, BORDERLINE_TICKETS, CLEAR_TICKETS

groq = groq_client()
GROQ_MODEL = os.getenv("MODEL")

try:
    gemini = gemini_client()
    # Gemini's Flash model name changes often - verify instead of trusting a name written down months ago.
    flash_models = list_gemini_models()
    GEMINI_MODEL = os.getenv("GEMINI_MODEL") or (flash_models[0] if flash_models else None)
    print("Gemini models available:", flash_models)
    print("using:", GEMINI_MODEL)
except Exception as e:
    gemini, GEMINI_MODEL = None, None
    print("Gemini not configured yet:", e)

try:
    ollama = ollama_client()
    OLLAMA_MODEL = "llama3.2:3b"
except Exception as e:
    ollama = None
    print("Ollama not available:", e)


Gemini models available: ['models/gemini-2.5-flash', 'models/gemini-2.5-flash-image', 'models/gemini-2.5-flash-lite', 'models/gemini-2.5-flash-native-audio-latest', 'models/gemini-2.5-flash-native-audio-preview-09-2025', 'models/gemini-2.5-flash-native-audio-preview-12-2025', 'models/gemini-2.5-flash-preview-tts', 'models/gemini-3-flash-preview', 'models/gemini-3.1-flash-image', 'models/gemini-3.1-flash-image-preview', 'models/gemini-3.1-flash-lite', 'models/gemini-3.1-flash-lite-image', 'models/gemini-3.1-flash-lite-preview', 'models/gemini-3.1-flash-live-preview', 'models/gemini-3.1-flash-tts-preview', 'models/gemini-3.5-flash', 'models/gemini-3.5-flash-lite', 'models/gemini-3.6-flash', 'models/gemini-3.7-flash', 'models/gemini-3.8-flash', 'models/gemini-3.8-flash-lite-tts', 'models/gemini-3.8-flash-tts', 'models/gemini-flash-latest', 'models/gemini-flash-lite-latest', 'models/gemini-omni-1.1-flash', 'models/gemini-omni-flash-preview']
using: gemini-3.8-flash


## 2. The messages format, properly

You have used this shape since `hello.py`: a list of role/content pairs. Two things worth seeing deliberately.

- The **system** message sets who the model is. Drop it and the model falls back to a generic default persona.
- **Order and role matter** — a system instruction buried as a second user message is weaker than a real system message.

In [7]:
system = "You are a terse assistant for a retail bank. Answer in one short sentence."
user = "What is an EMI?"

with_system = ask(groq, GROQ_MODEL, system, user)
without_system = ask(groq, GROQ_MODEL, "", user)

print("WITH system message:\n ", with_system["text"])
print("\nWITHOUT a system message:\n ", without_system["text"])


WITH system message:
  EMI stands for Equated Monthly Installment, a monthly payment plan for buying goods or services on credit.

WITHOUT a system message:
  EMI can refer to different things, but here are a few common meanings:

1. **Electric Motor Industry**: EMI can refer to the Electric Motor Industry, which is a sector that designs, manufactures, and distributes electric motors, drives, and control systems for various applications, including industrial, commercial, and residential uses.
2. **Emissions, Noise, and Interference**: In the context of electronics and audio engineering, EMI stands for Emissions, Noise, and Interference. It refers to the unwanted electromagnetic radiation or noise that can be emitted by electronic devices, such as radio transmitters, audio equipment, and other electronic systems.
3. **Economic Monopoly**: In economics, EMI can refer to an Economic Monopoly, where a single company or entity has a significant market share and dominates the industry, often

*Question 1 — What changed between the two answers, and which field in `.env` would you edit to make this run against Gemini instead of Groq?*

## 3. Same prompt, three providers

Same system prompt, same question, three providers. Compare the text and the token usage. This is the entire point of today: the code around this cell does not change — only which client and model name go in.

In [8]:
prompt = "In one sentence, what is the difference between a workflow and an agent?"
results = {}
for label, client, model in [("groq", groq, GROQ_MODEL), ("gemini", gemini, GEMINI_MODEL), ("ollama", ollama, "llama3.2:3b" if ollama else None)]:
    if client is None or model is None:
        print(f"--- {label}: skipped (not configured)\n"); continue
    try:
        r = ask(client, model, "You are a concise AI engineering tutor.", prompt)
        results[label] = r
        print(f"--- {label} ({model})")
        print(" ", r["text"].strip())
        print(f"   tokens: prompt={r['prompt_tokens']} completion={r['completion_tokens']} total={r['total_tokens']}\n")
    except Exception as e:
        print(f"--- {label}: error ({type(e).__name__}: {str(e)[:100]})\n")

--- groq (llama3.2:3b)
  A workflow is a series of tasks or processes that are executed in a specific order, whereas an agent is a software entity that autonomously interacts with the environment, making decisions and taking actions to achieve a goal.
   tokens: prompt=48 completion=43 total=91

--- gemini (gemini-3.8-flash)
  A **workflow** follows a predefined
   tokens: prompt=25 completion=7 total=221

--- ollama (llama3.2:3b)
  A workflow is a series of tasks or processes that are executed in a specific order, whereas an agent is a software entity that autonomously interacts with the environment, making decisions and taking actions to achieve a goal.
   tokens: prompt=48 completion=43 total=91



*Question 2 — Which provider used the fewest total tokens for the same question? Does that necessarily mean it is the cheapest or the best choice — what else would you need to know (hint: Session 10)?*

## 4. Sampling: temperature, top_p, max_tokens, stop

- **temperature** (0 to roughly 2) controls how much randomness goes into picking the next token. 0 is as close to deterministic as a provider allows; higher values let the model pick less-likely tokens more often.
- **top_p** is an alternative way to control the same thing — restrict the choice to the smallest set of tokens whose probabilities add up to `top_p`. Most labs here use temperature alone; the two are rarely combined.
- **max_tokens** caps the length of the reply. Set it too low on a reasoning model and you can get an empty answer, as you saw in Session 10.
- **stop** ends generation the moment a given string appears — useful for forcing a short, structured answer.

In [9]:
short = ask(groq, GROQ_MODEL, "You are a helpful assistant.", "Explain what a vector database is.", max_tokens=15)
enough = ask(groq, GROQ_MODEL, "You are a helpful assistant.", "Explain what a vector database is.", max_tokens=150)
print("max_tokens=15:\n ", short["text"], "\n")
print("max_tokens=150:\n ", enough["text"][:300])


max_tokens=15:
  A vector database is a type of database that stores and manages vectors, which 

max_tokens=150:
  A vector database is a type of database that stores and manages vectors, which are mathematical representations of objects in a high-dimensional space. These vectors can represent various types of data, such as images, text, audio, or other types of multimedia content.

In a traditional relational d


In [10]:
r = ask(groq, GROQ_MODEL, "You are a helpful assistant. Answer in a short paragraph.",
        "List three benefits of retrieval-augmented generation.", max_tokens=200, stop=["\n"])
print(r["text"])


Retrieval-augmented generation (RAG) offers several benefits. Firstly, it enables more accurate and informative content creation by leveraging existing knowledge and data, reducing the need for manual research and fact-checking. Secondly, RAG can generate content at a faster pace than traditional methods, making it ideal for applications where timely content is crucial, such as news articles or social media posts. Lastly, RAG can help reduce the risk of misinformation by relying on verifiable sources and fact-checked information, resulting in more trustworthy and reliable content.


## 5. The temperature sweep — where it actually matters

Three clear tickets, three borderline ones. Run the same ticket through the classifier at several temperatures and watch what happens. A routing decision like this should usually be made at temperature 0 — you do not want "billing" vs "technical" to depend on a coin flip.

In [11]:
TEMPERATURES = [0.0, 0.3, 0.7, 1.0]
RUNS_PER_TEMPERATURE = 3

if groq is None:
    print("skipped (no Groq client)")
else:
    for tid, text in BORDERLINE_TICKETS:
        print(f"\n=== {tid} ===\n{text}\n")
        for temp in TEMPERATURES:
            labels = []
            for _ in range(RUNS_PER_TEMPERATURE):
                result = classify_ticket(groq, GROQ_MODEL, text, temperature=temp)
                labels.append(result["label"] if result["valid"] else f"INVALID:{result['label']!r}")
            stable = len(set(labels)) == 1
            print(f"  temperature={temp:<4} {'stable' if stable else 'DRIFTED':8} {labels}")



=== borderline_1 ===
I upgraded my plan but I'm still being charged the old price and the new features aren't showing up either.

  temperature=0.0  stable   ['technical', 'technical', 'technical']
  temperature=0.3  DRIFTED  ['technical', 'technical', 'billing']
  temperature=0.7  DRIFTED  ['technical', 'technical', 'billing']
  temperature=1.0  DRIFTED  ['billing', 'technical', 'billing']

=== borderline_2 ===
Your app keeps logging me out and now I can't tell if I'm still being billed for the plan I signed up for.

  temperature=0.0  stable   ['technical', 'technical', 'technical']
  temperature=0.3  stable   ['technical', 'technical', 'technical']
  temperature=0.7  stable   ['technical', 'technical', 'technical']
  temperature=1.0  stable   ['technical', 'technical', 'technical']

=== borderline_3 ===
I'm not sure who to ask, but something seems wrong with my account since the last update.

  temperature=0.0  stable   ['general', 'general', 'general']
  temperature=0.3  stable   

*Question 3 — At which temperature, if any, did the label start to change between runs on the same ticket? What would you set temperature to for a routing decision in production, and why?*
*Question 4 — Run the same sweep on one of the CLEAR_TICKETS. Does it drift as much as the borderline ones? What does that tell you about when temperature actually matters?*

## 6. Switching provider — three lines, no code change

This is the payoff. Open `.env`. Comment the three Groq lines, uncomment the three Gemini (or Ollama) lines — or just point `BASE_URL`/`API_KEY`/`MODEL` at a different provider. Nothing in `providers_utils.py` or this notebook changes. Re-run the cell in section 3 after switching and compare.

In [12]:
# After editing .env, restart the kernel (Kernel > Restart), then re-run section 1's setup cell
# and this cell to see the SAME code now talking to a different provider.
import importlib, providers_utils
importlib.reload(providers_utils)
from providers_utils import ask
print("BASE_URL now:", os.getenv("BASE_URL"))
print("MODEL now:", os.getenv("MODEL"))

BASE_URL now: http://localhost:11434/v1
MODEL now: llama3.2:3b


## 7. Reflection

Answer questions 1–4 above, one or two lines each.

1. The system message made the answer specific and concise for a retail-bank context, giving a clear one-sentence definition of EMI, while without it the model returned an invalid/empty response (**). To switch from Groq to Gemini, edit the MODEL field in .env and use the Gemini configuration/API key as provided in the setup.

2. Groq used the fewest reported total tokens among the providers that successfully returned a response, with 236 total tokens (97 prompt + 139 completion). Gemini returned an error and Ollama returned a model-not-found error, so token usage was not available for them. Fewer tokens do not necessarily mean it is the cheapest or best choice; we also need to consider token pricing, model quality, capabilities, latency, and provider reliability.

3. For borderline_1, the result stayed stable at 0.0, 0.3, and 0.7, but drifted at 1.0 with an invalid result. For borderline_2, the result was stable at 0.0 and 0.3, but drifted at 0.7 and 1.0. borderline_3 remained stable at all temperatures. For production routing, use temperature 0 because routing should be deterministic and consistent rather than influenced by randomness.

4. The clear-ticket sweep should generally be more stable than the borderline tickets because the intent is more obvious. This shows that temperature matters more when the model is uncertain or the input is ambiguous; higher temperature can make borderline routing decisions less consistent.


Then run the checkpoint from the terminal: `uv run pytest tests/test_s11.py`